# Aplicação do tokenizador BPE

Este notebook demonstra a implementação de Byte Pair Encoding em nível de bytes. Ele procura um corpus em `dados/corpus_bpe/`. Caso a pasta não exista, utiliza um texto curto apenas para demonstrar o funcionamento do algoritmo.


In [ ]:
from pathlib import Path
import json
import sys

def localizar_raiz() -> Path:
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "01-tokenizacao-bpe").exists() and (candidato / "dados").exists():
            return candidato
    raise FileNotFoundError("Não foi possível localizar a raiz do repositório.")

ROOT = localizar_raiz()
MODULO = ROOT / "01-tokenizacao-bpe"
sys.path.insert(0, str(MODULO))

from bpe_tokenizer import BPETokenizer


In [ ]:
def carregar_corpus_json(pasta: Path) -> str:
    textos = []
    if not pasta.exists():
        return ""

    for arquivo in sorted(pasta.glob("*.json")):
        try:
            dados = json.loads(arquivo.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            continue

        if isinstance(dados, list):
            textos.extend(
                item.get("text", "")
                for item in dados
                if isinstance(item, dict) and item.get("text")
            )
        elif isinstance(dados, dict) and dados.get("text"):
            textos.append(dados["text"])

    return " ".join(textos)

corpus = carregar_corpus_json(ROOT / "dados" / "corpus_bpe")
if not corpus:
    corpus = (
        "Processamento de linguagem natural transforma texto em representações "
        "que podem ser analisadas por algoritmos de aprendizado de máquina. "
    ) * 20

print(f"Caracteres no corpus: {len(corpus):,}")


In [ ]:
tokenizer = BPETokenizer()
tokenizer.train(corpus, vocab_size=300)

amostra = corpus[:250]
ids = tokenizer.encode(amostra)
reconstruido = tokenizer.decode(ids)

print("Amostra original:")
print(amostra)
print("\nPrimeiros IDs:", ids[:40])
print("\nTexto reconstruído corretamente:", reconstruido == amostra)
print(f"Taxa de compressão da amostra: {tokenizer.compression_ratio(amostra):.2f}x")
print(f"Merges aprendidos: {len(tokenizer.merges)}")


## Interpretação

O BPE reduz a quantidade de unidades necessárias para representar padrões frequentes do corpus. Em textos maiores e repetitivos, sequências de bytes recorrentes são agrupadas em novos tokens, o que produz uma representação mais compacta do que a sequência bruta de bytes.
